In [5]:
# ============================================================
# FULL SCRIPT: MOUNT DRIVE + FIND PROJECT + PRINT TREE
# ============================================================

from google.colab import drive
import os
from pathlib import Path

# ------------------------------------------------------------
# 1. MOUNT DRIVE
# ------------------------------------------------------------
drive.mount("/content/drive")


# ------------------------------------------------------------
# 2. FIND WebKnoGraph AUTOMATICALLY
# ------------------------------------------------------------
def find_project(root="/content/drive/MyDrive", target="WebKnoGraph"):
    for current_root, dirs, files in os.walk(root):
        if target in dirs:
            return os.path.join(current_root, target)
    return None


project_path = find_project()

if project_path is None:
    raise ValueError(
        "WebKnoGraph folder not found in your Drive. Check the name or location."
    )

print(f"\nFound project at:\n{project_path}\n")


# ------------------------------------------------------------
# 3. PRINT FILE TREE
# ------------------------------------------------------------
def print_file_tree(root, max_depth=None, show_hidden=False):
    root_path = Path(root)

    def visible_children(path):
        children = list(path.iterdir())
        if not show_hidden:
            children = [c for c in children if not c.name.startswith(".")]
        return sorted(children, key=lambda p: (not p.is_dir(), p.name.lower()))

    def walk(path, prefix="", depth=0):
        if max_depth is not None and depth > max_depth:
            return

        children = visible_children(path)
        for i, child in enumerate(children):
            is_last = i == len(children) - 1
            branch = "└── " if is_last else "├── "

            if child.is_dir():
                try:
                    count = len(visible_children(child))
                except:
                    count = "?"
                print(f"{prefix}{branch}{child.name}/  ({count} items)")
                extension = "    " if is_last else "│   "
                walk(child, prefix + extension, depth + 1)
            else:
                try:
                    size = child.stat().st_size
                except:
                    size = "?"
                print(f"{prefix}{branch}{child.name}  ({size} bytes)")

    print(f"{root_path.name}/")
    walk(root_path)


# Print full tree
print_file_tree(project_path)


# ------------------------------------------------------------
# 4. SAVE TREE TO FILE (OPTIONAL)
# ------------------------------------------------------------
def save_tree_to_file(root, output_file="file_tree.txt"):
    lines = []
    root_path = Path(root)

    def visible_children(path):
        return sorted(
            [c for c in path.iterdir() if not c.name.startswith(".")],
            key=lambda p: (not p.is_dir(), p.name.lower()),
        )

    def walk(path, prefix=""):
        children = visible_children(path)
        for i, child in enumerate(children):
            is_last = i == len(children) - 1
            branch = "└── " if is_last else "├── "

            if child.is_dir():
                line = f"{prefix}{branch}{child.name}/"
                lines.append(line)
                extension = "    " if is_last else "│   "
                walk(child, prefix + extension)
            else:
                lines.append(f"{prefix}{branch}{child.name}")

    lines.append(f"{root_path.name}/")
    walk(root_path)

    output_path = os.path.join(root, output_file)
    with open(output_path, "w") as f:
        f.write("\n".join(lines))

    print(f"\nTree saved to: {output_path}")


# Save it
save_tree_to_file(project_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Found project at:
/content/drive/MyDrive/WebKnoGraph

WebKnoGraph/
├── assets/  (15 items)
│   ├── 03_link_graph.png  (89964 bytes)
│   ├── 04_graphsage_01.png  (201695 bytes)
│   ├── 04_graphsage_02.png  (125956 bytes)
│   ├── bmc-brand-logo.png  (4328 bytes)
│   ├── crawler_ui.png  (287949 bytes)
│   ├── embeddings_ui.png  (278369 bytes)
│   ├── fcse_logo.png  (10282 bytes)
│   ├── internal-linking-seo-roi-cropped.png  (54609 bytes)
│   ├── kalicube.com.png  (73462 bytes)
│   ├── pagerank_ui.png  (177570 bytes)
│   ├── product_roadmap.png  (80647 bytes)
│   ├── test_completed_1.png  (6670 bytes)
│   ├── test_completed_2.png  (37427 bytes)
│   ├── WebKnoGraph.png  (272145 bytes)
│   └── WL_logo.png  (47425 bytes)
├── data/  (6 items)
│   ├── crawled_data_parquet/  (1 items)
│   │   └── crawl_date=2025-06-28/  (1303 items)
│   │       ├── 1751112554.parquet 

# Check original links for inserting

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/WebKnoGraph")
RESULTS_DIR = PROJECT_ROOT / "results_2026"

BASELINE_FILE = RESULTS_DIR / "link_graph_edges.csv"

AUTOMATIC_ROOT = RESULTS_DIR / "automatic_led"
EXPERT_ROOT = RESULTS_DIR / "expert_led"

EXPECTED_K = 240

VALID_STRATEGY_DIRS = {
    "folder_batches": "Folder",
    "high_batches": "High",
    "low_batches": "Low",
    "mixed_batches": "Mixed",
    "random_batches": "Random",
}

OUTPUT_DIR = RESULTS_DIR / "updated_graph_repair_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def detect_edge_columns(df):
    from_candidates = ["FROM", "from", "From", "source", "Source", "src", "from_url"]
    to_candidates = ["TO", "to", "To", "target", "Target", "dst", "to_url"]

    from_col = next((c for c in from_candidates if c in df.columns), None)
    to_col = next((c for c in to_candidates if c in df.columns), None)

    if from_col is None or to_col is None:
        raise ValueError(f"No edge columns found. Columns: {list(df.columns)}")

    return from_col, to_col


def read_edges_exact(path):
    df = pd.read_csv(path)
    from_col, to_col = detect_edge_columns(df)

    edges = df[[from_col, to_col]].copy()
    edges.columns = ["FROM", "TO"]

    # Repo identity only: strip whitespace, no lowercasing, no slash removal.
    edges["FROM"] = edges["FROM"].astype(str).str.strip()
    edges["TO"] = edges["TO"].astype(str).str.strip()

    edges = edges[(edges["FROM"] != "") & (edges["TO"] != "")]

    return edges


def get_led_type(path):
    parts = [p.lower() for p in Path(path).parts]
    if "automatic_led" in parts:
        return "automatic"
    if "expert_led" in parts:
        return "expert"
    return "unknown"


def get_strategy_key(path):
    parts = [p.lower() for p in Path(path).parts]
    for folder in VALID_STRATEGY_DIRS:
        if folder in parts:
            return folder
    return "unknown"


def extract_batch_id(path):
    name = Path(path).stem
    import re

    patterns = [
        r"_graph_(\d+)$",
        r"_batch_(\d+)$",
        r"_(\d+)$",
    ]

    for pattern in patterns:
        m = re.search(pattern, name)
        if m:
            return int(m.group(1))

    return None


def discover_updated_graph_files():
    files = []

    for root in [AUTOMATIC_ROOT, EXPERT_ROOT]:
        for strategy_dir in VALID_STRATEGY_DIRS:
            d = root / strategy_dir
            if not d.exists():
                continue

            files.extend(d.glob("*updated_link_graph*.csv"))

    return sorted(
        files,
        key=lambda p: (
            get_led_type(p),
            get_strategy_key(p),
            extract_batch_id(p) or 999,
            str(p),
        ),
    )


baseline_edges_df = read_edges_exact(BASELINE_FILE)
baseline_edge_set = set(map(tuple, baseline_edges_df[["FROM", "TO"]].to_numpy()))

print("Baseline raw rows:", len(baseline_edges_df))
print("Baseline unique edges:", len(baseline_edge_set))

rows = []
problem_details = []

for fp in discover_updated_graph_files():
    updated_edges_df = read_edges_exact(fp)
    updated_edge_set = set(map(tuple, updated_edges_df[["FROM", "TO"]].to_numpy()))

    added_edges = updated_edge_set - baseline_edge_set
    removed_edges = baseline_edge_set - updated_edge_set

    tail_240 = updated_edges_df.tail(EXPECTED_K).copy()
    tail_edges = list(map(tuple, tail_240[["FROM", "TO"]].to_numpy()))
    tail_unique_edges = set(tail_edges)

    tail_duplicate_rows = len(tail_edges) - len(tail_unique_edges)
    tail_existing_in_baseline = [e for e in tail_edges if e in baseline_edge_set]
    tail_unique_new = tail_unique_edges - baseline_edge_set

    row = {
        "file_path": str(fp),
        "led_type": get_led_type(fp),
        "strategy": get_strategy_key(fp),
        "strategy_label": VALID_STRATEGY_DIRS.get(get_strategy_key(fp), "Unknown"),
        "batch_id": extract_batch_id(fp),
        "baseline_raw_rows": len(baseline_edges_df),
        "baseline_unique_edges": len(baseline_edge_set),
        "updated_raw_rows": len(updated_edges_df),
        "updated_unique_edges": len(updated_edge_set),
        "raw_row_difference": len(updated_edges_df) - len(baseline_edges_df),
        "unique_edge_difference": len(updated_edge_set) - len(baseline_edge_set),
        "added_unique_edges": len(added_edges),
        "removed_unique_edges": len(removed_edges),
        "tail_240_rows": len(tail_edges),
        "tail_240_unique_edges": len(tail_unique_edges),
        "tail_240_duplicate_rows": tail_duplicate_rows,
        "tail_240_existing_in_baseline": len(tail_existing_in_baseline),
        "tail_240_unique_new_edges": len(tail_unique_new),
        "status": "ok" if len(added_edges) == EXPECTED_K else "needs_review",
    }

    rows.append(row)

    if row["status"] == "needs_review":
        for edge in tail_existing_in_baseline[:50]:
            problem_details.append(
                {
                    "file_path": str(fp),
                    "problem": "tail_edge_already_in_baseline",
                    "FROM": edge[0],
                    "TO": edge[1],
                }
            )

        tail_df = pd.DataFrame(tail_edges, columns=["FROM", "TO"])
        dup_df = tail_df[tail_df.duplicated(["FROM", "TO"], keep=False)]

        for _, dup_row in dup_df.head(50).iterrows():
            problem_details.append(
                {
                    "file_path": str(fp),
                    "problem": "duplicate_inside_tail_240",
                    "FROM": dup_row["FROM"],
                    "TO": dup_row["TO"],
                }
            )

audit_df = pd.DataFrame(rows)
problem_details_df = pd.DataFrame(problem_details)

audit_path = OUTPUT_DIR / "updated_graph_240_audit.csv"
problem_path = OUTPUT_DIR / "updated_graph_240_problem_edges.csv"

audit_df.to_csv(audit_path, index=False)
problem_details_df.to_csv(problem_path, index=False)

print("\nSaved audit to:", audit_path)
print("Saved problem details to:", problem_path)

print("\nSummary by status:")
display(
    audit_df.groupby(["led_type", "strategy_label", "status"])
    .size()
    .reset_index(name="files")
)

print("\nProblem files:")
display(
    audit_df[audit_df["status"] == "needs_review"][
        [
            "led_type",
            "strategy_label",
            "batch_id",
            "raw_row_difference",
            "unique_edge_difference",
            "added_unique_edges",
            "removed_unique_edges",
            "tail_240_duplicate_rows",
            "tail_240_existing_in_baseline",
            "tail_240_unique_new_edges",
            "file_path",
        ]
    ]
)

Baseline raw rows: 122066
Baseline unique edges: 122066

Saved audit to: /content/WebKnoGraph/results_2026/updated_graph_repair_audit/updated_graph_240_audit.csv
Saved problem details to: /content/WebKnoGraph/results_2026/updated_graph_repair_audit/updated_graph_240_problem_edges.csv

Summary by status:


,led_type,strategy_label,status,files
0,automatic,Folder,needs_review,10
1,automatic,High,ok,10
2,automatic,Low,needs_review,1
3,automatic,Low,ok,9
4,automatic,Mixed,needs_review,9
5,automatic,Mixed,ok,1
6,automatic,Random,needs_review,8
7,automatic,Random,ok,2
8,expert,Folder,needs_review,2
9,expert,High,needs_review,2



Problem files:


,led_type,strategy_label,batch_id,raw_row_difference,unique_edge_difference,added_unique_edges,removed_unique_edges,tail_240_duplicate_rows,tail_240_existing_in_baseline,tail_240_unique_new_edges,file_path
0,automatic,Folder,1,232,232,232,0,0,8,232,/content/WebKnoGraph/results_2026/automatic_le...
1,automatic,Folder,2,232,232,232,0,0,8,232,/content/WebKnoGraph/results_2026/automatic_le...
2,automatic,Folder,3,230,230,230,0,0,10,230,/content/WebKnoGraph/results_2026/automatic_le...
3,automatic,Folder,4,228,228,228,0,0,12,228,/content/WebKnoGraph/results_2026/automatic_le...
4,automatic,Folder,5,226,226,226,0,0,14,226,/content/WebKnoGraph/results_2026/automatic_le...
5,automatic,Folder,6,234,234,234,0,0,6,234,/content/WebKnoGraph/results_2026/automatic_le...
6,automatic,Folder,7,224,224,224,0,0,16,224,/content/WebKnoGraph/results_2026/automatic_le...
7,automatic,Folder,8,226,226,226,0,0,14,226,/content/WebKnoGraph/results_2026/automatic_le...
8,automatic,Folder,9,227,227,227,0,0,13,227,/content/WebKnoGraph/results_2026/automatic_le...
9,automatic,Folder,10,224,224,224,0,0,16,224,/content/WebKnoGraph/results_2026/automatic_le...


# SSA Analysis

In [8]:
# ============================================================
# WebKnoGraph SSA / Delta Semantic Coherence
# Repo already cloned version
#
# Core correction:
# - Preserve repo edge identity for graph comparison.
# - Do NOT lowercase, strip trailing slashes, remove fragments,
#   or remove self-links when deciding whether an edge is new.
# - Use canonical URL variants only for embedding lookup fallback.
#
# This avoids SSA-specific over-cleaning that can make files with
# 240 added links appear to have fewer effective additions.
#
# Paper logic:
# - Use the realized updated graph files as stored.
# - Keep k = 240 as the intended intervention budget.
# - Report effective added edges as diagnostics.
# - Warn if any updated graph is not a monotonic expansion.
# ============================================================

!pip -q install pandas numpy tqdm pyarrow

from pathlib import Path
import re
import json
import ast
import warnings

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ============================================================
# 1. Configuration
# ============================================================

PROJECT_ROOT = Path("/content/WebKnoGraph")

RESULTS_DIR = PROJECT_ROOT / "results_2026"
BASELINE_FILE = RESULTS_DIR / "link_graph_edges.csv"

AUTOMATIC_ROOT = RESULTS_DIR / "automatic_led"
EXPERT_ROOT = RESULTS_DIR / "expert_led"

OUTPUT_DIR = RESULTS_DIR / "ssa_outputs_site_intervention_only_repo_identity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_K = 240
STRICT_K = False
EPSILON = 1e-9

VALID_STRATEGY_DIRS = {
    "folder_batches": "Folder",
    "high_batches": "High",
    "low_batches": "Low",
    "mixed_batches": "Mixed",
    "random_batches": "Random",
}

EDGE_FROM_COLS = [
    "FROM",
    "from",
    "From",
    "source",
    "Source",
    "SOURCE",
    "src",
    "SRC",
    "from_url",
    "source_url",
    "u",
]

EDGE_TO_COLS = [
    "TO",
    "to",
    "To",
    "target",
    "Target",
    "TARGET",
    "dst",
    "DST",
    "to_url",
    "target_url",
    "v",
]

EMB_URL_COLS = [
    "url",
    "URL",
    "Url",
    "page_url",
    "Page URL",
    "page",
    "node",
    "node_id",
]

EMB_VEC_COLS = [
    "embedding",
    "embeddings",
    "vector",
    "vectors",
    "page_embedding",
    "content_embedding",
]

OUTPUT_COLUMNS = [
    "led_type",
    "dataset",
    "strategy",
    "range",
    "batch_name",
    "file_path",
    "candidate_unique_edges",
    "n_edges_added",
    "removed_edges",
    "from_overlap",
    "to_overlap",
    "edge_overlap",
    "ssa_before",
    "ssa_after",
    "delta_ssa",
    "mean_sim_added",
    "median_sim_added",
    "std_sim_added",
    "added_edges_covered",
    "added_edges_total",
    "added_coverage",
    "updated_edges_covered",
    "updated_edges_total",
    "updated_coverage",
]

# ============================================================
# 2. Repo inspection
# ============================================================

print("\n=== Repo inspection ===")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Repo not found at {PROJECT_ROOT}. Clone it first or update PROJECT_ROOT."
    )

if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"Missing results_2026 directory: {RESULTS_DIR}")

print(f"Project root: {PROJECT_ROOT}")
print(f"results_2026 exists: {RESULTS_DIR.exists()}")
print(f"Baseline exists: {BASELINE_FILE.exists()} -> {BASELINE_FILE}")

print("\nTop level results_2026 entries:")
for p in sorted(RESULTS_DIR.iterdir()):
    print(" -", p.name)

print("\nautomatic_led entries:")
if AUTOMATIC_ROOT.exists():
    for p in sorted(AUTOMATIC_ROOT.iterdir()):
        print(" -", p.name)
else:
    print("Missing:", AUTOMATIC_ROOT)

print("\nexpert_led entries:")
if EXPERT_ROOT.exists():
    for p in sorted(EXPERT_ROOT.iterdir()):
        print(" -", p.name)
else:
    print("Missing:", EXPERT_ROOT)

print("\nExpected SSA strategy folders:")
for root in [AUTOMATIC_ROOT, EXPERT_ROOT]:
    print(f"\nRoot: {root}")
    for folder_name in VALID_STRATEGY_DIRS:
        d = root / folder_name
        csv_count = len(list(d.glob("*.csv"))) if d.exists() else 0
        print(f" - {folder_name}: exists={d.exists()}, csv_files={csv_count}")

# ============================================================
# 3. Helper functions
# ============================================================


def detect_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def clean_edge_url(value):
    """
    Edge identity cleaner.

    Important:
    This preserves the repo graph identity as much as possible.
    Only whitespace is removed.

    Do not lowercase.
    Do not strip trailing slash.
    Do not remove fragments.
    Do not canonicalize.
    Do not remove self-links here.
    """
    if pd.isna(value):
        return ""
    return str(value).strip()


def canonical_lookup_variants(value):
    """
    Embedding lookup variants.

    This is only used to find embeddings.
    It must not change graph edge identity or added-edge counts.
    """
    if pd.isna(value):
        return []

    value = str(value).strip()
    if value == "":
        return []

    candidates = [
        value,
        value.rstrip("/"),
        value.lower(),
        value.lower().rstrip("/"),
        value.split("#")[0],
        value.split("#")[0].rstrip("/"),
        value.split("#")[0].lower(),
        value.split("#")[0].lower().rstrip("/"),
    ]

    variants = []
    seen = set()

    for v in candidates:
        if v and v not in seen:
            variants.append(v)
            seen.add(v)

    return variants


def standardize_edge_columns(df, source_path=None):
    from_col = detect_col(df, EDGE_FROM_COLS)
    to_col = detect_col(df, EDGE_TO_COLS)

    if from_col is None or to_col is None:
        raise ValueError(
            f"No valid edge columns found in {source_path}. "
            f"Columns were: {list(df.columns)}"
        )

    out = df[[from_col, to_col]].copy()
    out.columns = ["FROM", "TO"]

    out["FROM"] = out["FROM"].apply(clean_edge_url)
    out["TO"] = out["TO"].apply(clean_edge_url)

    out = out[(out["FROM"] != "") & (out["TO"] != "")]

    # Preserve repo graph edge identity, but remove exact duplicate rows.
    out = out.drop_duplicates().reset_index(drop=True)

    return out


def load_edges(path):
    path = Path(path)

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path)

    return standardize_edge_columns(df, source_path=str(path))


def safe_cosine_sim(vec_a, vec_b, eps=EPSILON):
    if vec_a is None or vec_b is None:
        return np.nan

    vec_a = np.asarray(vec_a, dtype=np.float32)
    vec_b = np.asarray(vec_b, dtype=np.float32)

    if vec_a.ndim != 1 or vec_b.ndim != 1:
        return np.nan

    if vec_a.shape[0] != vec_b.shape[0]:
        return np.nan

    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)

    if norm_a < eps or norm_b < eps:
        return np.nan

    return float(np.dot(vec_a, vec_b) / (norm_a * norm_b))


def parse_embedding_value(value):
    if isinstance(value, np.ndarray):
        return value.astype(np.float32)

    if isinstance(value, list):
        return np.asarray(value, dtype=np.float32)

    if isinstance(value, str):
        s = value.strip()

        if s.startswith("[") and s.endswith("]"):
            try:
                return np.asarray(json.loads(s), dtype=np.float32)
            except Exception:
                try:
                    return np.asarray(ast.literal_eval(s), dtype=np.float32)
                except Exception:
                    pass

        s = s.strip("[]")
        parts = re.split(r"[,\s]+", s)
        parts = [p for p in parts if p]

        try:
            return np.asarray(parts, dtype=np.float32)
        except Exception:
            return np.array([], dtype=np.float32)

    try:
        return np.asarray(value, dtype=np.float32)
    except Exception:
        return np.array([], dtype=np.float32)


def find_embedding_files(project_root):
    candidates = []
    allowed_suffixes = {".csv", ".parquet", ".pkl", ".pickle"}

    skip_parts = {
        ".git",
        "ssa_outputs_site_intervention_only",
        "ssa_outputs_site_intervention_only_repo_identity",
        "__pycache__",
    }

    for path in project_root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in allowed_suffixes:
            continue

        parts_lower = {p.lower() for p in path.parts}
        if any(skip in parts_lower for skip in skip_parts):
            continue

        name = path.name.lower()
        full = str(path).lower()

        looks_embedding_related = (
            "embedding" in name
            or "embeddings" in name
            or "url_embeddings" in full
            or "content_embedding" in name
            or "page_embedding" in name
        )

        if looks_embedding_related:
            candidates.append(path)

    return sorted(candidates)


def add_embedding_to_lookup(lookup, url, vec):
    """
    Adds exact URL and fallback variants for embedding lookup.

    This increases coverage without changing graph edge identity.
    """
    if url is None:
        return

    exact = str(url).strip()
    if exact == "":
        return

    for v in canonical_lookup_variants(exact):
        if v and v not in lookup:
            lookup[v] = vec


def get_embedding(lookup, url):
    """
    Exact lookup first, then canonical variants.

    This is only for vector retrieval.
    """
    if url in lookup:
        return lookup[url]

    for v in canonical_lookup_variants(url):
        if v in lookup:
            return lookup[v]

    return None


def load_embedding_lookup(embedding_files):
    lookup = {}
    failures = []

    for path in tqdm(embedding_files, desc="Loading embedding files"):
        path = Path(path)

        try:
            if path.suffix.lower() == ".parquet":
                df = pd.read_parquet(path)

            elif path.suffix.lower() == ".csv":
                df = pd.read_csv(path)

            elif path.suffix.lower() in [".pkl", ".pickle"]:
                obj = pd.read_pickle(path)

                if isinstance(obj, dict):
                    for k, v in obj.items():
                        url = clean_edge_url(k)
                        vec = parse_embedding_value(v)

                        if url and vec.ndim == 1 and len(vec) > 0:
                            add_embedding_to_lookup(lookup, url, vec)

                    continue

                if isinstance(obj, pd.DataFrame):
                    df = obj
                else:
                    failures.append(
                        {
                            "file": str(path),
                            "error": f"Unsupported pickle object type: {type(obj)}",
                        }
                    )
                    continue
            else:
                continue

            url_col = detect_col(df, EMB_URL_COLS)
            vec_col = detect_col(df, EMB_VEC_COLS)

            if url_col is None:
                failures.append(
                    {
                        "file": str(path),
                        "error": f"Missing URL column. Columns: {list(df.columns)}",
                    }
                )
                continue

            if vec_col is not None:
                for _, row in df.iterrows():
                    url = clean_edge_url(row[url_col])
                    if not url:
                        continue

                    vec = parse_embedding_value(row[vec_col])
                    if vec.ndim == 1 and len(vec) > 0:
                        add_embedding_to_lookup(lookup, url, vec)

            else:
                possible_vector_cols = [c for c in df.columns if c != url_col]
                numeric_df = df[possible_vector_cols].apply(
                    pd.to_numeric,
                    errors="coerce",
                )

                numeric_cols = [
                    c
                    for c in possible_vector_cols
                    if numeric_df[c].notna().mean() > 0.95
                ]

                if len(numeric_cols) < 10:
                    failures.append(
                        {
                            "file": str(path),
                            "error": (
                                "No single embedding column and not enough "
                                "numeric vector columns."
                            ),
                        }
                    )
                    continue

                for idx, row in df.iterrows():
                    url = clean_edge_url(row[url_col])
                    if not url:
                        continue

                    vec = numeric_df.loc[idx, numeric_cols].to_numpy(dtype=np.float32)
                    if vec.ndim == 1 and len(vec) > 0:
                        add_embedding_to_lookup(lookup, url, vec)

        except Exception as e:
            failures.append({"file": str(path), "error": str(e)})

    return lookup, pd.DataFrame(failures)


def compute_alignment_stats(edges_df, emb_lookup):
    sims = []
    total = len(edges_df)

    for row in edges_df.itertuples(index=False):
        sim = safe_cosine_sim(
            get_embedding(emb_lookup, row.FROM),
            get_embedding(emb_lookup, row.TO),
        )

        if not np.isnan(sim):
            sims.append(sim)

    covered = len(sims)

    return {
        "mean_sim": float(np.mean(sims)) if covered > 0 else np.nan,
        "median_sim": float(np.median(sims)) if covered > 0 else np.nan,
        "std_sim": float(np.std(sims, ddof=1)) if covered > 1 else np.nan,
        "covered_edges": int(covered),
        "total_edges": int(total),
        "coverage": float(covered / total) if total > 0 else np.nan,
    }


def get_added_edges(baseline_df, candidate_df):
    """
    Uses exact repo edge identity after whitespace-only cleaning.
    """
    merged = candidate_df.merge(
        baseline_df,
        on=["FROM", "TO"],
        how="left",
        indicator=True,
    )

    added_df = (
        merged[merged["_merge"] == "left_only"][["FROM", "TO"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    return added_df


def compute_edge_overlap(edges_df, baseline_nodes):
    if len(edges_df) == 0:
        return np.nan, np.nan, np.nan

    from_known = edges_df["FROM"].isin(baseline_nodes)
    to_known = edges_df["TO"].isin(baseline_nodes)
    both_known = from_known & to_known

    return (
        float(from_known.mean()),
        float(to_known.mean()),
        float(both_known.mean()),
    )


def get_led_type(path):
    parts = [p.lower() for p in Path(path).parts]

    if "automatic_led" in parts:
        return "automatic"

    if "expert_led" in parts:
        return "expert"

    return "unknown"


def get_strategy_key(path):
    parts = [p.lower() for p in Path(path).parts]

    for folder_name in VALID_STRATEGY_DIRS:
        if folder_name in parts:
            return folder_name

    return "unknown"


def get_strategy_label(path):
    key = get_strategy_key(path)
    return VALID_STRATEGY_DIRS.get(key, "Unknown")


def extract_batch_id(path):
    name = Path(path).stem

    patterns = [
        r"_graph_(\d+)$",
        r"_batch_(\d+)$",
        r"_(\d+)$",
    ]

    for pattern in patterns:
        m = re.search(pattern, name)
        if m:
            return int(m.group(1))

    return None


def discover_intervention_files():
    files = []

    for root in [AUTOMATIC_ROOT, EXPERT_ROOT]:
        if not root.exists():
            continue

        for strategy_dir in VALID_STRATEGY_DIRS:
            d = root / strategy_dir

            if not d.exists():
                continue

            strategy_files = []

            for pattern in [
                "*updated_link_graph*.csv",
                "*updated_link_graph*.parquet",
                "*link_graph*.csv",
                "*link_graph*.parquet",
            ]:
                strategy_files.extend(d.glob(pattern))

            for fp in strategy_files:
                name = fp.name.lower()

                banned_terms = [
                    "overall",
                    "tracker",
                    "summary",
                    "metric",
                    "metrics",
                    "multi_range",
                    "authority",
                    "per_run",
                    "schema_matched",
                    "results_",
                ]

                if any(term in name for term in banned_terms):
                    continue

                files.append(fp)

    files = sorted(
        set(files),
        key=lambda p: (
            get_led_type(p),
            get_strategy_key(p),
            extract_batch_id(p) if extract_batch_id(p) is not None else 999999,
            str(p),
        ),
    )

    return files


def diagnose_added_edges_count(baseline_df, candidate_df):
    """
    Diagnostic count using exact repo edge identity.
    """
    baseline_edges = set(zip(baseline_df["FROM"], baseline_df["TO"]))
    candidate_edges = set(zip(candidate_df["FROM"], candidate_df["TO"]))

    added = candidate_edges - baseline_edges
    removed = baseline_edges - candidate_edges

    return {
        "baseline_unique_edges": len(baseline_edges),
        "candidate_unique_edges": len(candidate_edges),
        "added_edges": len(added),
        "removed_edges": len(removed),
        "net_edge_change": len(candidate_edges) - len(baseline_edges),
    }


# ============================================================
# 4. Load baseline graph
# ============================================================

print("\n=== Loading baseline graph ===")

if not BASELINE_FILE.exists():
    raise FileNotFoundError(f"Baseline graph not found: {BASELINE_FILE}")

baseline_df = load_edges(BASELINE_FILE)
baseline_nodes = set(baseline_df["FROM"]).union(set(baseline_df["TO"]))

print(f"Baseline file: {BASELINE_FILE}")
print(f"Baseline unique edges: {len(baseline_df):,}")
print(f"Baseline unique nodes: {len(baseline_nodes):,}")
display(baseline_df.head())

# ============================================================
# 5. Discover and load embeddings
# ============================================================

print("\n=== Discovering embedding files ===")

embedding_files = find_embedding_files(PROJECT_ROOT)

print(f"Embedding candidate files found: {len(embedding_files):,}")
for p in embedding_files[:30]:
    print(" -", p.relative_to(PROJECT_ROOT))

if len(embedding_files) > 30:
    print(f" ... plus {len(embedding_files) - 30} more")

if not embedding_files:
    raise FileNotFoundError(
        "No embedding files found in the cloned repo. SSA requires URL/page embeddings."
    )

print("\n=== Loading embeddings ===")

embedding_lookup, embedding_failures_df = load_embedding_lookup(embedding_files)

print(f"Loaded embedding lookup entries: {len(embedding_lookup):,}")

if len(embedding_lookup) == 0:
    display(embedding_failures_df.head(30))
    raise RuntimeError("No usable embeddings were loaded.")

# ============================================================
# 6. Baseline semantic coherence
# ============================================================

print("\n=== Computing baseline semantic coherence ===")

baseline_stats = compute_alignment_stats(baseline_df, embedding_lookup)
SSA_BASELINE = baseline_stats["mean_sim"]

print("Baseline SSA summary:")
print(f" - ssa_before: {SSA_BASELINE}")
print(f" - covered edges: {baseline_stats['covered_edges']:,}")
print(f" - total edges: {baseline_stats['total_edges']:,}")
print(f" - coverage: {baseline_stats['coverage']:.2%}")

if np.isnan(SSA_BASELINE):
    raise RuntimeError(
        "Baseline SSA is NaN. This usually means the URLs in the graph "
        "do not match the URLs in the embedding files."
    )

# ============================================================
# 7. Discover intervention graph files
# ============================================================

print("\n=== Discovering intervention graph files ===")

intervention_files = discover_intervention_files()

print(f"Intervention graph files found: {len(intervention_files):,}")

for p in intervention_files:
    print(
        f" - led={get_led_type(p):9s} "
        f"strategy={get_strategy_label(p):7s} "
        f"batch={str(extract_batch_id(p)):>4s} "
        f"path={p.relative_to(PROJECT_ROOT)}"
    )

if not intervention_files:
    raise FileNotFoundError(
        "No intervention graph files found under automatic_led/*_batches "
        "or expert_led/*_batches."
    )

# ============================================================
# 8. Evaluate all intervention graphs
# ============================================================

print("\n=== Evaluating SSA for intervention graphs ===")

results = []
failures = []
warnings_rows = []
audit_rows = []
diagnostics_rows = []

for fp in tqdm(intervention_files, desc="Evaluating intervention graphs"):
    fp = Path(fp)

    led_type = get_led_type(fp)
    strategy_key = get_strategy_key(fp)
    strategy_label = get_strategy_label(fp)
    batch_id = extract_batch_id(fp)

    try:
        candidate_df = load_edges(fp)

        candidate_unique_edges = len(candidate_df)

        from_overlap, to_overlap, edge_overlap = compute_edge_overlap(
            candidate_df,
            baseline_nodes,
        )

        diag = diagnose_added_edges_count(baseline_df, candidate_df)

        added_df = get_added_edges(baseline_df, candidate_df)
        n_edges_added = len(added_df)

        diagnostics_rows.append(
            {
                "file_path": str(fp),
                "led_type": led_type,
                "strategy": strategy_key,
                "strategy_label": strategy_label,
                "batch_id": batch_id,
                **diag,
            }
        )

        audit_rows.append(
            {
                "file_path": str(fp),
                "led_type": led_type,
                "strategy": strategy_key,
                "strategy_label": strategy_label,
                "batch_id": batch_id,
                "candidate_unique_edges": candidate_unique_edges,
                "n_edges_added": n_edges_added,
                "removed_edges": diag["removed_edges"],
                "from_overlap": from_overlap,
                "to_overlap": to_overlap,
                "edge_overlap": edge_overlap,
                "status": "accepted",
                "message": "",
            }
        )

        if diag["removed_edges"] != 0:
            message = (
                f"Updated graph is not a monotonic expansion: "
                f"{diag['removed_edges']} baseline edges are missing."
            )

            warnings_rows.append(
                {
                    "file_path": str(fp),
                    "batch_name": fp.stem,
                    "led_type": led_type,
                    "strategy": strategy_key,
                    "strategy_label": strategy_label,
                    "batch_id": batch_id,
                    "warning": message,
                    **diag,
                }
            )

            if STRICT_K:
                raise ValueError(message)

        if n_edges_added != EXPECTED_K:
            message = (
                f"Expected {EXPECTED_K} added edges, found {n_edges_added} "
                f"using repo-preserving edge identity."
            )

            warnings_rows.append(
                {
                    "file_path": str(fp),
                    "batch_name": fp.stem,
                    "led_type": led_type,
                    "strategy": strategy_key,
                    "strategy_label": strategy_label,
                    "batch_id": batch_id,
                    "warning": message,
                    **diag,
                }
            )

            if STRICT_K:
                raise ValueError(message)

        # Updated site graph according to repo identity.
        # Since the audit expects monotonic expansions, this equals baseline plus
        # effective new unique links. If removed_edges != 0, the warning above
        # makes the issue explicit.
        full_updated_df = (
            pd.concat([baseline_df, added_df], ignore_index=True)
            .drop_duplicates()
            .reset_index(drop=True)
        )

        added_stats = compute_alignment_stats(added_df, embedding_lookup)
        updated_stats = compute_alignment_stats(full_updated_df, embedding_lookup)

        ssa_after = updated_stats["mean_sim"]
        delta_ssa = (
            float(ssa_after - SSA_BASELINE)
            if not np.isnan(ssa_after) and not np.isnan(SSA_BASELINE)
            else np.nan
        )

        row = {
            "led_type": led_type,
            "dataset": "site_intervention_only",
            "strategy": strategy_key,
            "range": "site_intervention_only",
            "batch_name": fp.stem,
            "file_path": str(fp),
            "candidate_unique_edges": int(candidate_unique_edges),
            "n_edges_added": int(n_edges_added),
            "removed_edges": int(diag["removed_edges"]),
            "from_overlap": float(from_overlap),
            "to_overlap": float(to_overlap),
            "edge_overlap": float(edge_overlap),
            "ssa_before": float(SSA_BASELINE),
            "ssa_after": float(ssa_after) if not np.isnan(ssa_after) else np.nan,
            "delta_ssa": delta_ssa,
            "mean_sim_added": (
                float(added_stats["mean_sim"])
                if not np.isnan(added_stats["mean_sim"])
                else np.nan
            ),
            "median_sim_added": (
                float(added_stats["median_sim"])
                if not np.isnan(added_stats["median_sim"])
                else np.nan
            ),
            "std_sim_added": (
                float(added_stats["std_sim"])
                if not np.isnan(added_stats["std_sim"])
                else np.nan
            ),
            "added_edges_covered": int(added_stats["covered_edges"]),
            "added_edges_total": int(added_stats["total_edges"]),
            "added_coverage": (
                float(added_stats["coverage"])
                if not np.isnan(added_stats["coverage"])
                else np.nan
            ),
            "updated_edges_covered": int(updated_stats["covered_edges"]),
            "updated_edges_total": int(updated_stats["total_edges"]),
            "updated_coverage": (
                float(updated_stats["coverage"])
                if not np.isnan(updated_stats["coverage"])
                else np.nan
            ),
        }

        results.append(row)

    except Exception as e:
        failure_row = {
            "file_path": str(fp),
            "batch_name": fp.stem,
            "led_type": led_type,
            "strategy": strategy_key,
            "strategy_label": strategy_label,
            "batch_id": batch_id,
            "error": str(e),
        }

        failures.append(failure_row)

        audit_rows.append(
            {
                "file_path": str(fp),
                "led_type": led_type,
                "strategy": strategy_key,
                "strategy_label": strategy_label,
                "batch_id": batch_id,
                "candidate_unique_edges": np.nan,
                "n_edges_added": np.nan,
                "removed_edges": np.nan,
                "from_overlap": np.nan,
                "to_overlap": np.nan,
                "edge_overlap": np.nan,
                "status": "failed",
                "message": str(e),
            }
        )

# ============================================================
# 9. Save CSV outputs
# ============================================================

print("\n=== Saving outputs ===")

results_df = pd.DataFrame(results)

for col in OUTPUT_COLUMNS:
    if col not in results_df.columns:
        results_df[col] = np.nan

results_df = results_df[OUTPUT_COLUMNS]

results_df["batch_id"] = results_df["batch_name"].apply(
    lambda x: extract_batch_id(str(x))
)

results_df["strategy_label"] = results_df["strategy"].map(VALID_STRATEGY_DIRS)

results_df = results_df.sort_values(
    ["led_type", "strategy", "batch_id", "batch_name"],
    na_position="last",
).reset_index(drop=True)

final_cols = OUTPUT_COLUMNS + ["batch_id", "strategy_label"]
results_df = results_df[final_cols]

summary_df = (
    results_df.groupby(
        ["led_type", "dataset", "strategy", "strategy_label", "range"],
        dropna=False,
    )
    .agg(
        batches=("batch_name", "count"),
        mean_delta_ssa=("delta_ssa", "mean"),
        median_delta_ssa=("delta_ssa", "median"),
        std_delta_ssa=("delta_ssa", "std"),
        min_delta_ssa=("delta_ssa", "min"),
        max_delta_ssa=("delta_ssa", "max"),
        mean_ssa_after=("ssa_after", "mean"),
        mean_added_similarity=("mean_sim_added", "mean"),
        median_added_similarity=("mean_sim_added", "median"),
        mean_added_coverage=("added_coverage", "mean"),
        mean_edges_added=("n_edges_added", "mean"),
        min_edges_added=("n_edges_added", "min"),
        max_edges_added=("n_edges_added", "max"),
        total_removed_edges=("removed_edges", "sum"),
        max_removed_edges=("removed_edges", "max"),
    )
    .reset_index()
    .sort_values(["led_type", "strategy"])
)

baseline_summary_df = pd.DataFrame(
    [
        {
            "ssa_before": SSA_BASELINE,
            "baseline_edges_total": baseline_stats["total_edges"],
            "baseline_edges_covered": baseline_stats["covered_edges"],
            "baseline_coverage": baseline_stats["coverage"],
        }
    ]
)

failures_df = pd.DataFrame(failures)
warnings_df = pd.DataFrame(warnings_rows)
audit_df = pd.DataFrame(audit_rows)
diagnostics_df = pd.DataFrame(diagnostics_rows)

results_path = OUTPUT_DIR / "ssa_results_site_intervention_only_repo_identity.csv"
summary_path = (
    OUTPUT_DIR / "ssa_strategy_summary_site_intervention_only_repo_identity.csv"
)
baseline_path = (
    OUTPUT_DIR / "ssa_baseline_summary_site_intervention_only_repo_identity.csv"
)
warnings_path = OUTPUT_DIR / "ssa_warnings_site_intervention_only_repo_identity.csv"
failures_path = OUTPUT_DIR / "ssa_failures_site_intervention_only_repo_identity.csv"
audit_path = OUTPUT_DIR / "ssa_file_audit_site_intervention_only_repo_identity.csv"
diagnostics_path = OUTPUT_DIR / "ssa_added_edge_diagnostics_repo_identity.csv"
embedding_failures_path = OUTPUT_DIR / "ssa_embedding_load_failures_repo_identity.csv"

results_df.to_csv(results_path, index=False)
summary_df.to_csv(summary_path, index=False)
baseline_summary_df.to_csv(baseline_path, index=False)
warnings_df.to_csv(warnings_path, index=False)
failures_df.to_csv(failures_path, index=False)
audit_df.to_csv(audit_path, index=False)
diagnostics_df.to_csv(diagnostics_path, index=False)
embedding_failures_df.to_csv(embedding_failures_path, index=False)

print(f"Results saved to: {results_path}")
print(f"Summary saved to: {summary_path}")
print(f"Baseline summary saved to: {baseline_path}")
print(f"Warnings saved to: {warnings_path}")
print(f"Failures saved to: {failures_path}")
print(f"Audit saved to: {audit_path}")
print(f"Diagnostics saved to: {diagnostics_path}")
print(f"Embedding load failures saved to: {embedding_failures_path}")

print("\nRun summary:")
print(f"Validated intervention graphs: {len(results_df):,}")
print(f"Failures: {len(failures_df):,}")
print(f"Warnings: {len(warnings_df):,}")

print("\nResults preview:")
display(results_df.head(30))

print("\nStrategy summary:")
display(summary_df)

if len(warnings_df) > 0:
    print("\nWarnings preview:")
    display(warnings_df.head(30))

if len(failures_df) > 0:
    print("\nFailures preview:")
    display(failures_df.head(30))

print("\nDiagnostics preview:")
display(diagnostics_df.head(30))

# ============================================================
# 10. Paper ready export
# ============================================================

paper_summary_path = OUTPUT_DIR / "ssa_paper_strategy_values_repo_identity.csv"

paper_summary_df = summary_df[
    [
        "led_type",
        "strategy_label",
        "batches",
        "mean_delta_ssa",
        "std_delta_ssa",
        "mean_added_similarity",
        "mean_edges_added",
        "total_removed_edges",
        "max_removed_edges",
    ]
].copy()

paper_summary_df = paper_summary_df.rename(
    columns={
        "led_type": "Regime",
        "strategy_label": "Strategy",
        "batches": "Batches",
        "mean_delta_ssa": "Mean_Delta_SC",
        "std_delta_ssa": "Std_Delta_SC",
        "mean_added_similarity": "Mean_Added_Link_Similarity",
        "mean_edges_added": "Mean_Added_Edges",
        "total_removed_edges": "Total_Removed_Edges",
        "max_removed_edges": "Max_Removed_Edges",
    }
)

paper_summary_df.to_csv(paper_summary_path, index=False)

print(f"\nPaper summary saved to: {paper_summary_path}")
display(paper_summary_df)


=== Repo inspection ===
Project root: /content/WebKnoGraph
results_2026 exists: True
Baseline exists: True -> /content/WebKnoGraph/results_2026/link_graph_edges.csv

Top level results_2026 entries:
 - FineWeb_Data_Prep.ipynb
 - OVERALL_AVERAGES_TRACKER_AUTOMATIC_BA.csv
 - OVERALL_AVERAGES_TRACKER_AUTOMATIC_WWW.csv
 - OVERALL_AVERAGES_TRACKER_EXPERT_BA.csv
 - OVERALL_AVERAGES_TRACKER_EXPERT_WWW.csv
 - SSA_analysis.ipynb
 - WebKnoGraph_visualization.ipynb
 - automatic_led
 - base_file_types
 - deltas_BA_networkit_turbo.ipynb
 - deltas_Real_WWW_networkit.ipynb
 - expert_deltas_BA_networkit_turbo.ipynb
 - expert_deltas_Real_WWW_networkit.ipynb
 - expert_led
 - fineweb_500k_pages.csv
 - link_graph_edges.csv
 - ssa_outputs_site_intervention_only
 - ssa_outputs_site_intervention_only_repo_identity
 - ssa_results_stable.csv
 - updated_graph_repair_audit

automatic_led entries:
 - ba_results_automatic
 - fineweb_results_automatic
 - folder_batches
 - high_batches
 - high_boosters
 - low_batche

,FROM,TO
0,https://kalicube.com/,https://kalicube.com/case-studies/kalicube-pro...
1,https://kalicube.com/,https://kalicube.com/solutions/done-with-you-s...
2,https://kalicube.com/,https://kalicube.com/case-studies/brand-serp/u...
3,https://kalicube.com/,https://kalicube.com/learning-spaces/faq/
4,https://kalicube.com/,https://kalicube.com/book-discovery-call



=== Discovering embedding files ===
Embedding candidate files found: 1,789
 - data/url_embeddings/embeddings_batch_20250628_175716_1ee9a6e64d9a4af29a8808c8dd35196b.parquet
 - data/url_embeddings/embeddings_batch_20250628_175734_db13dda28f3d4b35bcecfae13323b756.parquet
 - data/url_embeddings/embeddings_batch_20250628_175735_92c293ecccc24865859c2c9c184d6a83.parquet
 - data/url_embeddings/embeddings_batch_20250628_175737_cd2c1647473d4eefabaf007deda5684d.parquet
 - data/url_embeddings/embeddings_batch_20250628_175739_ff78e5dc663640a586e0d7aaa2386cd9.parquet
 - data/url_embeddings/embeddings_batch_20250628_175740_ee000aed6ce24123ad4ca425e8977db3.parquet
 - data/url_embeddings/embeddings_batch_20250628_175752_27f5ab2551d846f9b7042a02e3e725ec.parquet
 - data/url_embeddings/embeddings_batch_20250628_175753_eda497fa269f4dfdb936383b2ea4b963.parquet
 - data/url_embeddings/embeddings_batch_20250628_175755_8437a5cdfc784e2594b7a2c37c1ea0bb.parquet
 - data/url_embeddings/embeddings_batch_20250628_17

Loading embedding files:   0%|          | 0/1789 [00:00<?, ?it/s]

Loaded embedding lookup entries: 3,413

=== Computing baseline semantic coherence ===
Baseline SSA summary:
 - ssa_before: 0.8272368688673822
 - covered edges: 106,862
 - total edges: 122,066
 - coverage: 87.54%

=== Discovering intervention graph files ===
Intervention graph files found: 60
 - led=automatic strategy=Folder  batch=   1 path=results_2026/automatic_led/folder_batches/240_folder_updated_link_graph_1.csv
 - led=automatic strategy=Folder  batch=   2 path=results_2026/automatic_led/folder_batches/240_folder_updated_link_graph_2.csv
 - led=automatic strategy=Folder  batch=   3 path=results_2026/automatic_led/folder_batches/240_folder_updated_link_graph_3.csv
 - led=automatic strategy=Folder  batch=   4 path=results_2026/automatic_led/folder_batches/240_folder_updated_link_graph_4.csv
 - led=automatic strategy=Folder  batch=   5 path=results_2026/automatic_led/folder_batches/240_folder_updated_link_graph_5.csv
 - led=automatic strategy=Folder  batch=   6 path=results_2026/auto

...

,file_path,batch_name,led_type,strategy,strategy_label,batch_id,warning,baseline_unique_edges,candidate_unique_edges,added_edges,removed_edges,net_edge_change
0,/content/WebKnoGraph/results_2026/automatic_le...,240_folder_updated_link_graph_1,automatic,folder_batches,Folder,1,"Expected 240 added edges, found 232 using repo...",122066,122298,232,0,232
1,/content/WebKnoGraph/results_2026/automatic_le...,240_folder_updated_link_graph_2,automatic,folder_batches,Folder,2,"Expected 240 added edges, found 232 using repo...",122066,122298,232,0,232
2,/content/WebKnoGraph/results_2026/automatic_le...,240_folder_updated_link_graph_3,automatic,folder_batches,Folder,3,"Expected 240 added edges, found 230 using repo...",122066,122296,230,0,230
3,/content/WebKnoGraph/results_2026/automatic_le...,240_folder_updated_link_graph_4,automatic,folder_batches,Folder,4,"Expected 240 added edges, found 228 using repo...",122066,122294,228,0,228
4,/content/WebKnoGraph/results_2026/automatic_le...,240_folder_updated_link_graph_5,automatic,folder_batches,Folder,5,"Expected 240 added edges, found 226 using repo...",122066,122292,226,0,226
5,/content/WebKnoGraph/results_2026/automatic_le...,240_folder_updated_link_graph_6,automatic,folder_batches,Folder,6,"Expected 240 added edges, found 234 using repo...",122066,122300,234,0,234
6,/content/WebKnoGraph/results_2026/automatic_le...,240_folder_updated_link_graph_7,automatic,folder_batches,Folder,7,"Expected 240 added edges, found 224 using repo...",122066,122290,224,0,224
7,/content/WebKnoGraph/results_2026/automatic_le...,240_folder_updated_link_graph_8,automatic,folder_batches,Folder,8,"Expected 240 added edges, found 226 using repo...",122066,122292,226,0,226
8,/content/WebKnoGraph/results_2026/automatic_le...,240_folder_updated_link_graph_9,automatic,folder_batches,Folder,9,"Expected 240 added edges, found 227 using repo...",122066,122293,227,0,227
9,/content/WebKnoGraph/results_2026/automatic_le...,240_folder_updated_link_graph_10,automatic,folder_batches,Folder,10,"Expected 240 added edges, found 224 using repo...",122066,122290,224,0,224



Diagnostics preview:


,file_path,led_type,strategy,strategy_label,batch_id,baseline_unique_edges,candidate_unique_edges,added_edges,removed_edges,net_edge_change
0,/content/WebKnoGraph/results_2026/automatic_le...,automatic,folder_batches,Folder,1,122066,122298,232,0,232
1,/content/WebKnoGraph/results_2026/automatic_le...,automatic,folder_batches,Folder,2,122066,122298,232,0,232
2,/content/WebKnoGraph/results_2026/automatic_le...,automatic,folder_batches,Folder,3,122066,122296,230,0,230
3,/content/WebKnoGraph/results_2026/automatic_le...,automatic,folder_batches,Folder,4,122066,122294,228,0,228
4,/content/WebKnoGraph/results_2026/automatic_le...,automatic,folder_batches,Folder,5,122066,122292,226,0,226
5,/content/WebKnoGraph/results_2026/automatic_le...,automatic,folder_batches,Folder,6,122066,122300,234,0,234
6,/content/WebKnoGraph/results_2026/automatic_le...,automatic,folder_batches,Folder,7,122066,122290,224,0,224
7,/content/WebKnoGraph/results_2026/automatic_le...,automatic,folder_batches,Folder,8,122066,122292,226,0,226
8,/content/WebKnoGraph/results_2026/automatic_le...,automatic,folder_batches,Folder,9,122066,122293,227,0,227
9,/content/WebKnoGraph/results_2026/automatic_le...,automatic,folder_batches,Folder,10,122066,122290,224,0,224



Paper summary saved to: /content/WebKnoGraph/results_2026/ssa_outputs_site_intervention_only_repo_identity/ssa_paper_strategy_values_repo_identity.csv


,Regime,Strategy,Batches,Mean_Delta_SC,Std_Delta_SC,Mean_Added_Link_Similarity,Mean_Added_Edges,Total_Removed_Edges,Max_Removed_Edges
0,automatic,Folder,10,-0.000177,0.000009,0.671693,228.3,0,0
1,automatic,High,10,-0.000292,0.000012,0.631136,240.0,0,0
2,automatic,Low,10,-0.000273,0.000016,0.634402,239.9,0,0
3,automatic,Mixed,10,-0.000265,0.000018,0.634778,238.4,0,0
4,automatic,Random,10,-0.000235,0.000015,0.612479,237.0,0,0
5,expert,Folder,2,-0.000124,0.000010,0.675710,229.0,0,0
6,expert,High,2,-0.000166,0.000063,0.661983,219.0,0,0
7,expert,Low,2,-0.000201,0.000101,0.633899,240.0,0,0
8,expert,Mixed,2,-0.000123,0.000070,0.627916,174.5,0,0
9,expert,Random,2,-0.000162,0.000111,0.621914,238.0,0,0
